In [87]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [88]:
test['keyword'].isnull().sum()
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


In [89]:
# import pandas as pd
# from ydata_profiling import ProfileReport
#
# # Membaca data CSV
#
# # Membuat ringkasan data otomatis
# profile = ProfileReport(train, title="Ringkasan Dataset CSV")
# profile.to_file("ringkasan_data_train.html")


In [90]:
id_train, id_test = train['id'], test['id']
TARGET = train['target']
SEED = 42
n_estimator = 500

In [91]:
# drop kolom
drop_col = ['id', 'location']
train_clean = train.drop(drop_col, axis=1)
test_clean = test.drop(drop_col, axis=1)

In [92]:
# ppenanganan data kosong
modus = train_clean['keyword'].mode()[0]
print(modus)

#imputasi
train_clean['keyword'] = train_clean['keyword'].fillna(modus)
test_clean['keyword'] = test_clean['keyword'].fillna(modus)

fatalities


In [93]:
# Basic Cleaning

import re
from nltk.corpus import stopwords
import nltk
import matplotlib.pyplot as plt

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hajar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [94]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

In [95]:
train_clean['clean_text'] = train_clean['text'].apply(clean_text)
test_clean['clean_text'] = test_clean['text'].apply(clean_text)

In [96]:
print(train_clean['clean_text'].describe())

count                                                  7613
unique                                                 7469
top       yearold boy charged manslaughter toddler repor...
freq                                                     10
Name: clean_text, dtype: object


In [97]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

X = train_clean['clean_text']
y = train_clean['target']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

In [98]:
#Embeddings

vectorizer = TfidfVectorizer(max_features=500)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vect = vectorizer.transform(X_val)

In [99]:
X_train_vec

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20509 stored elements and shape (6090, 500)>

In [100]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

lr = LogisticRegression(max_iter=500, random_state=SEED)
lr.fit(X_train_vec, y_train)

y_pred_lr = lr.predict(X_val_vect)
print("Logistic Regression")
print(classification_report(y_val, y_pred_lr))

Logistic Regression
              precision    recall  f1-score   support

           0       0.79      0.86      0.82       869
           1       0.79      0.69      0.74       654

    accuracy                           0.79      1523
   macro avg       0.79      0.78      0.78      1523
weighted avg       0.79      0.79      0.79      1523



In [101]:
#Random Forest

from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=n_estimator, max_depth=10 ,random_state=SEED)
rf.fit(X_train_vec, y_train)
print("Random Forest")
print(classification_report(y_val, rf.predict(X_val_vect)))

Random Forest
              precision    recall  f1-score   support

           0       0.65      0.98      0.78       869
           1       0.91      0.30      0.45       654

    accuracy                           0.68      1523
   macro avg       0.78      0.64      0.61      1523
weighted avg       0.76      0.68      0.64      1523



In [102]:
# XGBoost
import xgboost as xgb

# Encoding Target KLO targetnya kategorical
# y_train_enc = y_train.map({'negatif' : 0, 'positif': 1})
# y_val_enc = y_val.map({'negatif' : 0, 'positif': 1})

xgb_model = xgb.XGBClassifier(n_estimators=n_estimator, max_depth=20 ,random_state=SEED, n_jobs=-1)
xgb_model.fit(X_train_vec, y_train) #sesuaikan X nya jika di encode

print('XGBoost')
print(classification_report(y_val, xgb_model.predict(X_val_vect)))

XGBoost
              precision    recall  f1-score   support

           0       0.79      0.83      0.81       869
           1       0.76      0.70      0.73       654

    accuracy                           0.77      1523
   macro avg       0.77      0.76      0.77      1523
weighted avg       0.77      0.77      0.77      1523



In [103]:
# Evaluaste & Compare

from sklearn.metrics import f1_score
import numpy as np

models = {
    'Logistic' : lr.predict(X_val_vect),
    'RandomForest' : rf.predict(X_val_vect),
    'XGBoost' : xgb_model.predict(X_val_vect)
}

comparison = []
for name, y_pred in models.items():

    #Jalankan hanya jika targetnya kategorical
    # if name in ['Logistic', 'RandomForest', 'XGBoost']:
    #     y_pred_numeric = np.array([0 if p == 'negative' else 1 for p in y_pred])
    # else :
    #     y_pred_numeric = y_pred

    f1 = f1_score(y_val, y_pred)
    comparison.append({'Model' : name, 'F1': f1})

print(pd.DataFrame(comparison).sort_values(by='F1', ascending=False))

          Model        F1
0      Logistic  0.737612
2       XGBoost  0.726550
1  RandomForest  0.445727


In [104]:
sample_submission = pd.read_csv('sample_submission.csv')
sample_submission.head()

,id,target
0,0,0
1,2,0
2,3,0
3,9,0
4,11,0


In [107]:
sample_submission['target'] = np.argmax(lr.predict(X_val_vect))

In [108]:
sample_submission.head()

,id,target
0,0,0
1,2,0
2,3,0
3,9,0
4,11,0


In [110]:
sample_submission.to_csv('submission.csv', index=False)